<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/14-structured-streaming/05-stream-joins.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 14.5 — Stream-static and stream-stream joins

A streaming "orders" joined to a static "customers" dimension
(snapshot-at-start behavior made visible); then two synthetic
streams -- "orders" and "shipments" -- joined with a time-range
condition and watermarks on both sides.

In [ ]:
import os, sys, shutil, time, json
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("14.5-stream-joins")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## Stream-static: the static side is a SNAPSHOT, not live

In [ ]:
orders_stream = (spark.readStream.format("rate").option("rowsPerSecond", 5).load()
                 .withColumn("customer_id", (col("value") % 3).cast("int")))

customers = spark.createDataFrame(
    [(0, "alice"), (1, "bob"), (2, "carol")], ["customer_id", "name"])   # STATIC, read once

enriched = orders_stream.join(customers, "customer_id")

sq = enriched.writeStream.outputMode("append").format("memory").queryName("enriched_demo").start()
time.sleep(4)
print("joined output -- static side resolved at query start:")
spark.sql("SELECT * FROM enriched_demo ORDER BY timestamp DESC LIMIT 5").show()
sq.stop()

print("\nNOTE: if `customers` changed right now in the source system, this RUNNING query")
print("would NOT see the change -- it already has its snapshot (14.5's stale-static warning).")

## Stream-stream: two streams, a time-range condition, watermarks on both

In [ ]:
orders = (spark.readStream.format("rate").option("rowsPerSecond", 5).load()
          .withColumnRenamed("timestamp", "order_time")
          .withColumnRenamed("value", "order_id")
          .withWatermark("order_time", "10 seconds"))

# shipments: same underlying counter, so order_id lines up, but timestamped
# a few seconds LATER to simulate "shipped after ordered"
shipments = (spark.readStream.format("rate").option("rowsPerSecond", 5).load()
             .withColumn("ship_time", col("timestamp") + F.expr("INTERVAL 2 SECONDS"))
             .withColumnRenamed("value", "ship_order_id")
             .withWatermark("ship_time", "10 seconds")
             .drop("timestamp"))

joined = orders.join(
    shipments,
    F.expr("""
        order_id = ship_order_id AND
        ship_time BETWEEN order_time AND order_time + INTERVAL 5 SECONDS
    """))

join_q = (joined.writeStream.outputMode("append")
          .format("memory").queryName("join_demo").start())

time.sleep(10)
print("matched order/shipment pairs (inner stream-stream join):")
spark.sql("SELECT order_id, order_time, ship_time FROM join_demo ORDER BY order_time LIMIT 10").show(truncate=False)
join_q.stop()

## Rejected: stream-stream join with NO time-range condition

In [ ]:
from pyspark.sql.utils import AnalysisException
try:
    bad_join = orders.join(shipments, orders.order_id == shipments.ship_order_id)   # no time bound
    bad_q = bad_join.writeStream.outputMode("append").format("memory").queryName("bad_join").start()
    bad_q.awaitTermination(3)
except AnalysisException as e:
    print("rejected -- no time-range bound means Spark can't know when to give up buffering:")
    print(str(e).splitlines()[0])

In [ ]:
spark.stop()

## Exercises

1. Change the static `customers` DataFrame's `name` for customer_id=0
   AFTER the query has started once (in a fresh cell, reassign
   `customers` and re-run `.join()` — but the ALREADY RUNNING query
   won't see it). Confirm: does a NEW query see the update? Does the
   OLD running query?
2. Widen the time-range condition from `INTERVAL 5 SECONDS` to
   `INTERVAL 1 SECOND` (tighter). Do you see fewer matches? Why would
   a real "shipped after ordered" pipeline need to pick this window
   carefully?
3. Give `shipments` a much wider watermark (`"2 minutes"`) while
   `orders` keeps `"10 seconds"`. Does the join still work? What does
   14.5 say about each stream needing its OWN watermark, independently
   sized?
4. Convert the inner join to a left outer join
   (`orders.join(shipments, ..., "left_outer")`). After letting it run
   long enough for some orders' watermark to close without a shipment
   match, do you see null-filled rows for `ship_time`?
5. Try building the outer join the OTHER direction — static-left,
   stream-right for the stream-static case (swap `enriched =
   customers.join(orders_stream, "customer_id", "left_outer")`). Does
   Spark accept it? Reread the lesson's explanation for why only one
   direction is legal.

In [ ]:
# your exercise code here